# RTENSO2026: animated evolution of the stratospheric polar vortex

Day-by-day animation of the Northern Hemisphere polar vortex at 10 hPa in the E3SM RTENSO2026
hindcasts from the 1 Feb, 1 May and 1 Aug 2026 starts, next to ERA5, for the winter `WINTER`
(1 Oct - 30 Apr by default). The default is the 2026/27 winter, a forecast with no observations yet.
`WINTER = 2025` shows the observed 2025/26 winter against the 1 Feb 2026 start. Each frame is one day:

* **Top row**: 10-hPa geopotential height from ERA5 and the pooled ensemble mean of each start.
  Contours are the total height (every `STYLE["line"]` m) and shading the anomaly.
* **Bottom row**: the ERA5 1981-2010 climatology of the same day, then one **featured member** of
  each start. The ensemble mean smooths away the vortex dynamics (displacements, splits); a single
  member shows them. By default the featured member is the first member of that start with an SSW
  in the winter (`FEATURED` picks one by hand).
* **Vortex index plumes** (static, before the animation): U(60°N, 10 hPa), 10-hPa polar-cap
  temperature, the 10-hPa polar-cap height anomaly and the polar-cap sea-level pressure anomaly over the whole forecast period, the
  vortex counterpart of the Niño3.4 plume in notebook 8.
* **Timeline**: U(60°N, 10 hPa) of each start's ensemble mean (thick) and featured member (thin),
  ERA5 and its climatology (±1σ), drawn up to the frame day. Below zero is an SSW or the final warming.

**Anomalies and totals.** Model anomalies remove the Reanalysis 1981-2010 lead-time climatology of
10-hPa height (`rc.matched_climatology("Z10")`; Feb/Aug starts use `CLIM_METHOD`), interpolated from
months to days, plus the offset (~40 m) between the native daily `Z010` and the monthly `Z3` product
of the same runs (the ensemble mean of each month). Model totals add those anomalies to the ERA5 climatology, as in notebook 5, so the
contours show the forecast vortex without the model's mean bias. ERA5 anomalies remove its own
1981-2010 monthly climatology, interpolated to days.

**Read with notebook 9.** Three of its findings matter here. (1) E3SM's DJF vortex is weaker than
ERA5's (U(60°N, 10 hPa) about 23 vs 29 m/s) and varies about half as much between winters (σ about 4 vs
10 m/s); the map totals hide this (they add the anomaly to the ERA5 climatology), so the timeline shows
each start's own climatology. (2) For the Feb and Aug starts, which have no Reanalysis climatology,
the sign of the vortex anomaly changes between `lead_drift` and `target_month` (marked †; the plume
shows both). (3) Only some members have an SSW in a given winter; the featured member is one of them by
construction, not the typical member.

**Data.** Model: native `eam.h1` daily-mean `Z010`, remapped to 1° with `MAP_FILE` and cached for
Sep-May (`util/strat_case.model_daily_maps`). ERA5: the 00Z analysis at 10 hPa from the hourly
0.25° mirror (`ERA5_MIRROR`), averaged to 1° and cached (`era5_daily_maps`; extended automatically
when the mirror is). U(60°N, 10 hPa) comes from the notebook 9 zonal-mean caches and `ERA5/1x1/daily_zonal`.

Outputs: a GIF and a video (H.264 MP4 if ffmpeg has libx264, else VP9 WebM) and a static storyboard
under `rc.OUTPUT_ROOT/10_polar_vortex_animation/figures/`.

**Pooled ensembles:** as in notebooks 8 and 9 (`POOL`, 30 members per start; the bit-identical
1 Feb 2026 `clim` and `obs` runs count once, so that start has 20).

In [ ]:
%matplotlib inline

import subprocess
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.dates as mdates
import matplotlib.path as mpath
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

from IPython.display import Image, display
from matplotlib import animation
from matplotlib.colors import BoundaryNorm
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from cartopy.util import add_cyclic_point
from esp_lab.utils import colormap_utils as mycolors
from esp_lab.utils.filename_utils import figure_filename

from util import rtenso_case as rc
from util import strat_case as sc

warnings.filterwarnings("ignore", message="invalid value encountered")
warnings.filterwarnings("ignore", message="Mean of empty slice")

### Run controls

In [ ]:
NOTEBOOK = "10_polar_vortex_animation"   # outputs: rc.OUTPUT_ROOT/10_polar_vortex_animation/{figures,data}/
# "auto": reuse the map and climatology caches; "rebuild": re-read the model output and ERA5; "require": fail if missing
derivation_mode = "auto"
CLIM_METHOD = "lead_drift"          # Feb/Aug drift correction: "lead_drift" or "target_month"
WORKERS = 4                         # processes reading model output and ERA5 (cache builds only)

# Data locations (machine-specific)
OBSERVATION_ROOT = Path("/global/cfs/cdirs/e3sm/zhan391/data")                       # reference-data archive
ERA5_ZONAL_DIR = OBSERVATION_ROOT / "ERA5" / "1x1" / "daily_zonal"                    # ERA5 daily zonal means
ERA5_DAILY_DIR = OBSERVATION_ROOT / "ERA5" / "1x1" / "daily"                          # ERA5 daily maps (PSL)
ERA5_MIRROR = Path("/global/cfs/projectdirs/m3522/cmip6/ERA5")                        # hourly 0.25-degree ERA5
SIMULATION_ROOT = Path("/global/cfs/cdirs/e3sm/S2S2D/RTENSO2026/simulation")         # raw eam.h1 history
MAP_FILE = Path("/global/cfs/cdirs/e3sm/diagnostics/maps/map_ne30pg2_to_cmip6_180x360_aave.20200201.nc")

YEAR = 2026
INIT_MONTHS = (2, 5, 8)
MONTH_NAMES = {1: "Jan", 2: "Feb", 3: "Mar", 4: "Apr", 5: "May", 6: "Jun", 7: "Jul", 8: "Aug",
               9: "Sep", 10: "Oct", 11: "Nov", 12: "Dec"}
INIT_COLORS = {2: "#1f77b4", 5: "#2ca02c", 8: "#d62728"}
MODEL_PREFIX = rc.MODEL_PREFIX
POOL = {
    "Reanalysis-CLIM": f"{MODEL_PREFIX}_RTENSO2026_clim",
    "Reanalysis-OBS": f"{MODEL_PREFIX}_RTENSO2026_obs",
    "Reanalysis-NSIDC": f"{MODEL_PREFIX}_RTENSO2026_obs_ice_nsidc",
}

WINTER = 2026                       # year of the November: 2026 -> 2026/27 (forecast); 2025 -> 2025/26 (observed)
FRAMES = (f"{WINTER}-10-01", f"{WINTER + 1}-04-30")   # first and last frame day
FRAME_STEP = 2                      # days between frames
FEATURED = {}                       # {init month: "Reanalysis-CLIM:3"}; a missing start picks its first SSW member
STORYBOARD = [f"{WINTER}-11-15", f"{WINTER}-12-15", f"{WINTER + 1}-01-15", f"{WINTER + 1}-02-15",
              f"{WINTER + 1}-03-15", f"{WINTER + 1}-04-15"]
CLIM_YEARS = rc.CLIM_YEARS
ERA5_FIRST_YEAR = 1979              # ERA5 daily record read for the climatologies
SSW = {"sep_days": 20, "recover_days": 10}

# Vortex index plumes (static figure over the whole forecast period)
INDEX_SPAN = (f"{YEAR}-01-01", f"{YEAR + 2}-07-31")
SMOOTH_DAYS = 5                     # centred running mean of the daily indices (1: none)
INDEX_RANGE = (0.1, 0.9)            # member quantiles shaded

LAT_MIN = 30.0                      # map edge
STYLE = {"ci": 100.0, "range": 1200.0, "line": 400.0, "units": "m", "cmap": "blue2red",
         "name": "10-hPa geopotential height anomaly"}

# Animation output
FPS = 6                             # frames per second (FRAME_STEP days each)
HOLD = 6                            # extra copies of the last frame, so a looping GIF pauses at the end
ANIM_DPI = 100                      # 14 x 9.5 in at 100 dpi: 1400 x 950 px
ANIM_FORMATS = ("gif", "video")     # "video": H.264 MP4 if ffmpeg has libx264 (imageio-ffmpeg), else VP9 WebM
SHOW_INLINE = True                  # show the GIF in the notebook
fig_dpi = 200                       # static storyboard

OUTDIR = rc.output_dir(NOTEBOOK, "data")
FIGDIR = rc.figure_dir(NOTEBOOK)
WINTER_LABEL = f"{WINTER}/{str(WINTER + 1)[2:]}"

### Load

ERA5 maps (from the mirror, through its last complete day) and U(60°N, 10 hPa) with its daily
climatology; the model maps of each start (pooled), their drift-corrected anomalies and totals; the
model U(60°N, 10 hPa) from the notebook 9 caches. Everything is restricted to the frame days.

In [ ]:
%%time
frame_days = pd.date_range(FRAMES[0], FRAMES[1], freq=f"{FRAME_STEP}D")
all_days = pd.date_range(FRAMES[0], FRAMES[1], freq="D")


def nh(da):
    return da.sel(lat=slice(LAT_MIN - 0.01, 90.0))


# ERA5: U(60N, 10 hPa) and its smoothed daily climatology
era5_u = sc.era5_daily_zonal(ERA5_ZONAL_DIR, fields=("U",), plev=[10.0])["U"].sel(plev=10.0)
e_u60 = sc.lat_mean(era5_u, sc.U60)
clim_u60, std_u60 = sc.daily_climatology(e_u60, CLIM_YEARS)
obs_end = pd.Timestamp(e_u60["time"].values[-1])
doy = xr.DataArray(sc.noleap_doy(all_days.values), dims="time", coords={"time": all_days.values})
u60_clim_days, u60_std_days = clim_u60.sel(doy=doy).drop_vars("doy"), std_u60.sel(doy=doy).drop_vars("doy")

# ERA5 10-hPa height: 1981-2010 monthly climatology (e3sm_diags ERA5 zg) to days, and the observed days
z_clim_month = nh(rc.obs_monthly_climatology("Z10", mode=derivation_mode)).load()
z_clim = sc.daily_from_month_clim(z_clim_month, all_days.values)                     # (time, lat, lon)
obs_days = all_days[all_days <= obs_end]
if len(obs_days):
    z_obs = nh(sc.era5_daily_maps(ERA5_MIRROR, obs_days[0], obs_days[-1], mode=derivation_mode, workers=WORKERS))
    z_obs = z_obs.interp(lat=z_clim["lat"], lon=z_clim["lon"]) if not np.array_equal(z_obs["lon"], z_clim["lon"]) else z_obs
    obs_anom = z_obs - z_clim.sel(time=z_obs["time"])
    print(f"ERA5 10-hPa maps {obs_days[0]:%d %b %Y} - {obs_days[-1]:%d %b %Y}")
else:
    z_obs = obs_anom = None
    print(f"ERA5 ends {obs_end:%d %b %Y}: winter {WINTER_LABEL} not observed yet")

# Model: maps, anomalies and totals of the ensemble mean and the featured member; U(60N, 10 hPa)
fc = {}          # init -> dict of DataArrays on the model days within the frames
for init in INIT_MONTHS:
    zd = sc.pooled_daily_zonal(SIMULATION_ROOT, POOL, YEAR, init, mode=derivation_mode, workers=WORKERS)
    maps = sc.pooled_daily_maps(SIMULATION_ROOT, MAP_FILE, POOL, YEAR, init, mode=derivation_mode, workers=WORKERS)
    if zd is None or maps is None:
        print(f"[init {MONTH_NAMES[init]}] no model output")
        continue
    u60 = sc.lat_mean(zd["U"].sel(plev=10.0), sc.U60)                                   # (member, time)
    days = all_days[all_days.isin(pd.DatetimeIndex(maps["time"].values))]
    if not len(days):
        print(f"[init {MONTH_NAMES[init]}] no days in the frames")
        continue
    z = nh(maps.sel(time=days.values)).load()
    lead_clim = nh(rc.matched_climatology("Z10", init, method=CLIM_METHOD, mode=derivation_mode))
    # native daily Z010 vs the monthly Z3 product the climatology is made from (~40 m): add the
    # ensemble-mean offset of the same runs, month by month (Sep-May, the months the map cache keeps)
    mon = [rc.open_hindcast(prefix, YEAR, init, "Z10") for prefix in POOL.values()]
    mon = nh(xr.concat([m.drop_vars("member") for m in mon if m is not None], dim="member").mean("member"))
    dmean = sc.monthly_means(nh(maps).mean("member")).load()
    targets = sc.to_datetime64(mon["time"].values).astype("datetime64[M]").astype("datetime64[ns]") + np.timedelta64(14, "D")
    z_off = dmean.reindex(time=targets).rename(time="lead").assign_coords(lead=mon["lead"].values) - mon.drop_vars("time")
    clim_days = sc.daily_from_lead(lead_clim + z_off, YEAR, init, days.values)
    anom = z - clim_days
    # featured member: FEATURED, else the first SSW of the winter, else the weakest DJF vortex
    feat = FEATURED.get(init)
    if feat is None:
        first = []
        for mem in u60["member"].values:
            ev = [d for d in sc.ssw_central_dates(u60.sel(member=mem).values, u60["time"].values, **SSW)
                  if sc.winter_of(d) == WINTER]
            if ev:
                first.append((ev[0], str(mem)))
        if first:
            date, feat = min(first)
            why = f"first SSW of the winter ({date:%d %b %Y}); {len(first)} of {u60.sizes['member']} members have an SSW"
        else:
            djf = u60.sel(time=slice(f"{WINTER}-12-01", f"{WINTER + 1}-02-28")).mean("time")
            feat = str(djf["member"].values[int(djf.argmin())])
            why = "weakest DJF vortex (no member has an SSW this winter)"
    else:
        why = "set in FEATURED"
    feat_ssw = [d for d in sc.ssw_central_dates(u60.sel(member=feat).values, u60["time"].values, **SSW) if sc.winter_of(d) == WINTER]
    n_ssw = sum(any(sc.winter_of(d) == WINTER for d in sc.ssw_central_dates(u60.sel(member=m).values, u60["time"].values, **SSW))
                for m in u60["member"].values)
    fc[init] = {
        "days": days, "n": int(z.sizes["member"]), "featured": feat, "why": why,
        "mean_anom": anom.mean("member"), "mean_total": anom.mean("member") + z_clim.sel(time=days.values),
        "feat_anom": anom.sel(member=feat), "feat_total": anom.sel(member=feat) + z_clim.sel(time=days.values),
        "u60_mean": u60.mean("member"), "u60_feat": u60.sel(member=feat), "feat_ssw": feat_ssw, "n_ssw": n_ssw,
        "surrogate": init not in rc.REFERENCE["init_months"],
    }
    print(f"[init {MONTH_NAMES[init]} {YEAR}] {fc[init]['n']} members, {len(days)} days in the frames; "
          f"featured {feat}: {why}")

### Vortex index plumes

The polar-vortex counterpart of the Niño3.4 plume: daily indices over the whole forecast period,
smoothed with a `SMOOTH_DAYS` running mean. Each start's ensemble mean (thick), its `INDEX_RANGE`
member range (shading) and, for the raw quantities, its own lead-time climatology (dashed: what the
model drifts to), against ERA5 (black) and the ERA5 1981-2010 climatology ±1σ (grey).

* **U(60°N, 10 hPa)**: vortex strength; below zero is an SSW or the final warming. Ticks along
  the bottom are every member's SSW central dates (one row per start) and ERA5's (black).
* **T(60-90°N, 10 hPa)**: the warming of an SSW.
* **Polar-cap (60-90°N) height anomaly at 10 hPa** (m, grey: ERA5 ±1σ; positive is a weak vortex)
  and **sea-level pressure anomaly** (divided by the ERA5 daily standard deviation; the surface
  imprint: positive is a negative NAM). The height is shown in metres because its summer σ is a few
  tens of metres, which would turn the summer trend into several σ. Model anomalies remove the lead-time climatology (notebook 9, `CLIM_METHOD`).
  The native daily Z is ~40 m higher at 10 hPa than the monthly product the climatology is made
  from (online vs post-hoc vertical interpolation; U, T and PSL agree), so each run's ensemble-mean
  daily-minus-monthly offset is added to the climatology (`sc.daily_monthly_offset`).
  The Feb and Aug starts have no Reanalysis climatology; their `lead_drift` climatology borrows the
  mean May/Nov drift, which is poor at 10 hPa in the first summer months (the Aug start's lead-0 jump).

In [ ]:
%%time
LEVS = [10.0]


def vortex_indices(ds):
    return {"u60": sc.lat_mean(ds["U"].sel(plev=10.0), sc.U60),
            "t10": sc.lat_mean(ds["T"].sel(plev=10.0), sc.PCAP),
            "z10": sc.lat_mean(ds["Z"].sel(plev=10.0), sc.PCAP)}


def on_doy(clim, times):
    d = xr.DataArray(sc.noleap_doy(times), dims="time", coords={"time": times})
    return clim.sel(doy=d).drop_vars("doy")


# ERA5 indices and their smoothed daily climatologies (mean, std)
era5_z = sc.era5_daily_zonal(ERA5_ZONAL_DIR, fields=("U", "T", "Z"), plev=LEVS)
e_idx = vortex_indices(era5_z)
e_idx["slp"] = sc.era5_polar_cap_slp(ERA5_DAILY_DIR, years=(ERA5_FIRST_YEAR, YEAR))
e_iclim = {k: sc.daily_climatology(v, CLIM_YEARS) for k, v in e_idx.items()}

# model indices: raw, the lead-time climatology on the same days, and the anomaly
ref = {m: sc.monthly_zonal(rc.REFERENCE["case_prefix"], "Reanalysis", range(CLIM_YEARS[0], CLIM_YEARS[1] + 1), m,
                           mode=derivation_mode, workers=WORKERS).sel(plev=LEVS)
       for m in rc.REFERENCE["init_months"]}
e_month = sc.obs_month_climatology(era5_z[["U", "T", "Z"]], CLIM_YEARS)
m_idx = {}
for init in INIT_MONTHS:
    zd = sc.pooled_daily_zonal(SIMULATION_ROOT, POOL, YEAR, init, mode=derivation_mode, workers=WORKERS)
    if zd is None:
        continue
    # daily (native) Z differs from the monthly product by a near-constant offset: add it to the climatology
    mo = sc.pooled_monthly_zonal(POOL, YEAR, init, mode=derivation_mode, workers=WORKERS)
    z_off = sc.daily_monthly_offset(zd, mo, YEAR, init).sel(plev=LEVS)
    zd = zd.sel(plev=LEVS)
    lc = sc.lead_climatology(init, ref, e_month, method=CLIM_METHOD)
    lc["Z"] = lc["Z"] + z_off
    cd = sc.daily_from_lead(lc, YEAR, init, zd["time"].values).assign_coords(time=zd["time"])
    raw, clim = vortex_indices(zd), vortex_indices(cd)
    # surface: polar-cap sea-level pressure against the drift-corrected PSL climatology (rtenso_case)
    psl_clim = sc.lat_mean(rc.matched_climatology("PSL", init, method=CLIM_METHOD, mode=derivation_mode).mean("lon"), sc.PCAP)
    raw["slp"] = sc.lat_mean(zd["PSL"], sc.PCAP)
    clim["slp"] = sc.daily_from_lead(psl_clim, YEAR, init, zd["time"].values).assign_coords(time=zd["time"])
    alt = None
    if init not in ref:                       # Feb/Aug: lead_drift is a surrogate; target_month as the alternative
        lc_alt = sc.lead_climatology(init, ref, e_month, method="target_month")
        lc_alt["Z"] = lc_alt["Z"] + z_off
        clim_alt = vortex_indices(sc.daily_from_lead(lc_alt, YEAR, init, zd["time"].values).assign_coords(time=zd["time"]))
        pc_alt = sc.lat_mean(rc.matched_climatology("PSL", init, method="target_month", mode=derivation_mode).mean("lon"), sc.PCAP)
        clim_alt["slp"] = sc.daily_from_lead(pc_alt, YEAR, init, zd["time"].values).assign_coords(time=zd["time"])
        alt = {k: raw[k] - clim_alt[k] for k in ("z10", "slp")}
    m_idx[init] = {"raw": raw, "clim": clim, "anom": {k: raw[k] - clim[k] for k in raw}, "anom_alt": alt,
                   "ssw": [d for mem in raw["u60"]["member"].values
                           for d in sc.ssw_central_dates(raw["u60"].sel(member=mem).values, raw["u60"]["time"].values, **SSW)]}
obs_ssw = sc.ssw_central_dates(e_idx["u60"].values, e_idx["u60"]["time"].values, **SSW)

In [ ]:
PANELS = [("u60", "raw", "U(60°N, 10 hPa)", "m s$^{-1}$"),
          ("t10", "raw", "T(60-90°N, 10 hPa)", "K"),
          ("z10", "anom", "Polar-cap (60-90°N) height anomaly, 10 hPa", "m"),
          ("slp", "std", "Polar-cap (60-90°N) sea-level pressure anomaly (surface)", "σ")]
span = slice(*INDEX_SPAN)
span_days = pd.date_range(*INDEX_SPAN, freq="D").values


def smooth(da):
    return da.rolling(time=SMOOTH_DAYS, center=True, min_periods=1).mean() if SMOOTH_DAYS > 1 else da


fig, axes = plt.subplots(len(PANELS), 1, figsize=(15, 3.3 * len(PANELS)), sharex=True, gridspec_kw={"hspace": 0.18})
for ax, (key, kind, title, units) in zip(axes, PANELS):
    mean, std = e_iclim[key]
    # winters (DJF) shaded lightly
    for y in range(pd.Timestamp(INDEX_SPAN[0]).year - 1, pd.Timestamp(INDEX_SPAN[1]).year + 1):
        ax.axvspan(np.datetime64(f"{y}-12-01"), np.datetime64(f"{y + 1}-03-01"), color="tab:blue", alpha=0.04, lw=0)
    if kind == "raw":
        c, sd = on_doy(mean, span_days), on_doy(std, span_days)
        ax.fill_between(span_days, c - sd, c + sd, color="0.85", lw=0)
        ax.plot(span_days, c, color="0.45", lw=1.2)
    elif kind == "anom":
        sd = on_doy(std, span_days)
        ax.fill_between(span_days, -sd, sd, color="0.85", lw=0)
    else:
        ax.axhspan(-1, 1, color="0.88", lw=0)
    if key == "u60" or kind != "raw":
        ax.axhline(0, color="k", lw=0.8)
    for init, m in m_idx.items():
        color = INIT_COLORS[init]
        s = m["raw" if kind == "raw" else "anom"][key]
        if kind == "std":
            s = s / on_doy(std, s["time"].values)
        s = smooth(s).sel(time=span)
        if not s.sizes["time"]:
            continue
        t = s["time"].values
        q = s.quantile(list(INDEX_RANGE), "member")
        ax.fill_between(t, q.isel(quantile=0), q.isel(quantile=1), color=color, alpha=0.16, lw=0)
        ens = s.mean("member")
        ax.plot(t, ens, color=color, lw=2.3)
        ax.plot(t[0], float(ens[0]), "o", color=color, ms=7, mec="k", mew=0.6, zorder=5)
        if kind == "raw":
            ax.plot(t, m["clim"][key].sel(time=span), color=color, lw=1.1, ls="--", alpha=0.7)
        elif m["anom_alt"] is not None and key in m["anom_alt"]:
            a = m["anom_alt"][key]
            if kind == "std":
                a = a / on_doy(std, a["time"].values)
            ax.plot(t, smooth(a).sel(time=span).mean("member"), color=color, lw=1.6, ls=":")
    o = e_idx[key].sel(time=span)
    if o.sizes["time"]:
        if kind != "raw":
            o = o - on_doy(mean, o["time"].values)
        if kind == "std":
            o = o / on_doy(std, o["time"].values)
        ax.plot(o["time"].values, smooth(o), color="k", lw=2.2, zorder=6)
    ax.set_ylabel(units)
    ax.set_title(title, loc="left", fontweight="bold")
    ax.grid(alpha=0.25)

# SSW central dates: one row of ticks per start, ERA5 in black
ax = axes[0]
rows = [("ERA5", "k", [d for d in obs_ssw if pd.Timestamp(INDEX_SPAN[0]) <= d <= pd.Timestamp(INDEX_SPAN[1])])]
rows += [(f"{MONTH_NAMES[i]} start", INIT_COLORS[i], m["ssw"]) for i, m in m_idx.items()]
for k, (label, color, dates) in enumerate(rows):
    y = 0.03 + 0.05 * k
    ax.plot(dates, np.full(len(dates), y), "|", color=color, ms=9, mew=1.6, transform=ax.get_xaxis_transform(), zorder=7)
n_ssw = ", ".join(f"{label}: {len(d)}" for label, _, d in rows[1:])
ax.text(0.995, 0.97, f"SSWs (ticks): {n_ssw}", transform=ax.transAxes, ha="right", va="top", fontsize=9,
        bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.8})

axes[-1].set_xlim(np.datetime64(INDEX_SPAN[0]), np.datetime64(INDEX_SPAN[1]))
axes[-1].xaxis.set_major_locator(mdates.MonthLocator(bymonth=(1, 4, 7, 10)))
axes[-1].xaxis.set_minor_locator(mdates.MonthLocator())
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%b\n%Y"))
handles = [Line2D([], [], color="k", lw=2.2, label="ERA5"),
           Patch(color="0.85", label=f"ERA5 {CLIM_YEARS[0]}-{CLIM_YEARS[1]} ±1σ")]
handles += [Line2D([], [], color=INIT_COLORS[i], lw=2.3, marker="o", mec="k", ms=6,
                   label=f"1 {MONTH_NAMES[i]} {YEAR} start ({m_idx[i]['raw']['u60'].sizes['member']})") for i in m_idx]
handles += [Line2D([], [], color="0.3", lw=1.6, ls=":", label="Feb/Aug mean, target_month climatology"),
            Patch(color="0.5", alpha=0.3, label=f"member {INDEX_RANGE[0]:.0%}-{INDEX_RANGE[1]:.0%}"),
            Line2D([], [], color="0.4", lw=1.1, ls="--", label="E3SM lead-time climatology")]
axes[-1].legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, -0.32), ncol=4, frameon=False, fontsize=10)
fig.suptitle(f"Stratospheric polar vortex, E3SM RTENSO2026 {YEAR} starts and ERA5 (through {obs_end:%d %b %Y}); "
             f"{SMOOTH_DAYS}-day running means", x=0.125, ha="left", y=0.995, fontsize=15, fontweight="bold")
print(rc.save_figure(fig, NOTEBOOK, "RTENSO2026", f"y{YEAR}", "vortex_indices", "plume", dpi=fig_dpi))
plt.show()

### Frame layout

Two rows of polar maps (ERA5 and its climatology, then each start's ensemble mean above its featured
member) above the U(60°N, 10 hPa) timeline. `draw_frame` redraws only what changes, so the same function
draws the animation frames, the preview and the storyboard panels.

In [ ]:
FIG = {
    "size": (14.0, 9.6),
    "height_ratios": (1.0, 1.0, 0.045, 0.62),     # maps, maps, colorbar, timeline
    "hspace": 0.30, "wspace": 0.04,
    "left": 0.04, "right": 0.98, "top": 0.89, "bottom": 0.08,
    "suptitle": 16, "day": 20, "title": 10.5, "label": 10.5, "tick": 9.5, "legend": 9.5, "badge": 9.5,
    "coast_lw": 0.5, "empty": "0.93", "ens_lw": 2.6, "member_lw": 1.2, "obs_lw": 2.4, "faded_alpha": 0.22,
}
PROJ = ccrs.NorthPolarStereo(central_longitude=-90)    # North America at the bottom-left, Eurasia top-right
_theta = np.linspace(0, 2 * np.pi, 120)
CIRCLE = mpath.Path(np.vstack([np.sin(_theta), np.cos(_theta)]).T * 0.5 + [0.5, 0.5])


def levels_cmap():
    levels = np.arange(-STYLE["range"], STYLE["range"] + STYLE["ci"] / 2, STYLE["ci"])
    cmap = mycolors.blue2red_cmap(levels) if STYLE["cmap"] == "blue2red" else plt.get_cmap(STYLE["cmap"], len(levels) - 1)
    return levels, cmap, BoundaryNorm(levels, ncolors=cmap.N)


def setup_map(ax, title, color="k"):
    # extent as a square in projected coordinates reaching LAT_MIN: set_extent([-180, 180, ...]) collapses
    # the y range of a polar-stereographic map in cartopy 0.26
    x, y = PROJ.transform_point(0.0, LAT_MIN, ccrs.PlateCarree())
    r = float(np.hypot(x, y))
    ax.set_xlim(-r, r)
    ax.set_ylim(-r, r)
    ax.set_boundary(CIRCLE, transform=ax.transAxes)
    ax.add_feature(cfeature.LAND, facecolor="0.88", zorder=0.5)
    ax.coastlines(lw=FIG["coast_lw"], color="0.35", zorder=3)
    ax.gridlines(crs=ccrs.PlateCarree(), lw=0.3, color="0.5", alpha=0.5, xlocs=range(-180, 181, 45), ylocs=[40, 60, 80])
    ax.set_title(title, loc="center", fontsize=FIG["title"], color=color, fontweight="bold")
    return {"artists": [],
            "empty": ax.text(0.5, 0.5, "", transform=ax.transAxes, ha="center", va="center", fontsize=FIG["label"],
                             color="0.35", zorder=6, bbox={"facecolor": "white", "edgecolor": "0.6", "boxstyle": "round,pad=0.4"}),
            "badge": ax.text(0.5, -0.04, "", transform=ax.transAxes, ha="center", va="top", fontsize=FIG["badge"])}


def fill_map(ax, state, anom=None, total=None, empty="", badge="", badge_color="k"):
    # shade ``anom`` and contour ``total`` (lat, lon); a grey panel with ``empty`` when both are None
    for artist in state["artists"]:
        artist.remove()
    state["artists"] = []
    state["badge"].set_text(badge)
    state["badge"].set_color(badge_color)
    if anom is None and total is None:
        ax.set_facecolor(FIG["empty"])
        state["empty"].set_text(empty)
        state["empty"].set_visible(True)
        return
    ax.set_facecolor("white")
    state["empty"].set_visible(False)
    if anom is not None:
        levels, cmap, norm = levels_cmap()
        values, lon = add_cyclic_point(anom.values, coord=anom["lon"].values)
        state["artists"].append(ax.contourf(lon, anom["lat"].values, values, levels=levels, cmap=cmap, norm=norm,
                                            extend="both", transform=ccrs.PlateCarree(), zorder=1))
    if total is not None:
        values, lon = add_cyclic_point(total.values, coord=total["lon"].values)
        lines = np.arange(np.floor(np.nanmin(values) / STYLE["line"]) * STYLE["line"], np.nanmax(values) + STYLE["line"], STYLE["line"])
        state["artists"].append(ax.contour(lon, total["lat"].values, values, levels=lines, colors="k", linewidths=0.8,
                                           transform=ccrs.PlateCarree(), zorder=2))


def value_at(da, t):
    t = np.datetime64(pd.Timestamp(t), "ns")
    return da.sel(time=t) if t in da["time"].values else None


def u60_badge(value, prefix="U60"):
    # keyword arguments (badge, badge_color) of fill_map
    if value is None or not np.isfinite(value):
        return {"badge": "", "badge_color": "k"}
    return {"badge": f"{prefix}(10 hPa) {value:+.0f} m/s" + ("  easterly" if value < 0 else ""),
            "badge_color": "tab:red" if value < 0 else "k"}

In [ ]:
def build_figure():
    inits = [i for i in INIT_MONTHS if i in fc]
    ncol = 1 + len(inits)
    fig = plt.figure(figsize=FIG["size"])
    gs = fig.add_gridspec(4, ncol, height_ratios=FIG["height_ratios"], hspace=FIG["hspace"], wspace=FIG["wspace"],
                          left=FIG["left"], right=FIG["right"], top=FIG["top"], bottom=FIG["bottom"])
    maps = {}
    maps["obs"] = (fig.add_subplot(gs[0, 0], projection=PROJ), "ERA5", "k")
    maps["clim"] = (fig.add_subplot(gs[1, 0], projection=PROJ), f"ERA5 {CLIM_YEARS[0]}-{CLIM_YEARS[1]} climatology", "0.3")
    for k, init in enumerate(inits, start=1):
        maps[("mean", init)] = (fig.add_subplot(gs[0, k], projection=PROJ),
                                f"1 {MONTH_NAMES[init]} {YEAR} start: mean of {fc[init]['n']}{' †' if fc[init]['surrogate'] else ''}",
                                INIT_COLORS[init])
        maps[("feat", init)] = (fig.add_subplot(gs[1, k], projection=PROJ),
                                f"member {fc[init]['featured']}{' †' if fc[init]['surrogate'] else ''}", INIT_COLORS[init])
    state = {"maps": {key: (ax, setup_map(ax, title, color)) for key, (ax, title, color) in maps.items()}}

    levels, cmap, norm = levels_cmap()
    cax = fig.add_subplot(gs[2, :])
    pos = cax.get_position()
    cax.set_position([pos.x0 + pos.width * 0.25, pos.y0, pos.width * 0.5, pos.height])
    cbar = fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=cmap), cax=cax, orientation="horizontal", extend="both",
                        ticks=np.arange(-STYLE["range"], STYLE["range"] + 1e-6, 4 * STYLE["ci"]))
    cbar.set_label(f"{STYLE['name']} ({STYLE['units']}); contours: total height every {STYLE['line']:g} m",
                   fontsize=FIG["label"])
    cbar.ax.tick_params(labelsize=FIG["tick"])

    # timeline: static faded series, dynamic solid parts up to the frame day
    ax = fig.add_subplot(gs[3, :])
    t_all = all_days.values
    ax.fill_between(t_all, u60_clim_days - u60_std_days, u60_clim_days + u60_std_days, color="0.85", lw=0)
    ax.plot(t_all, u60_clim_days, color="0.45", lw=1.2)
    ax.axhline(0, color="k", lw=0.8)
    lines = {}
    for init in inits:
        color = INIT_COLORS[init]
        series = {}
        for key, lw, ls in (("u60_mean", FIG["ens_lw"], "-"), ("u60_feat", FIG["member_lw"], "--")):
            s = fc[init][key].sel(time=slice(all_days[0], all_days[-1]))
            ax.plot(s["time"].values, s.values, color=color, lw=lw * 0.7, ls=ls, alpha=FIG["faded_alpha"])
            series[key] = (s, ax.plot([], [], color=color, lw=lw, ls=ls, zorder=4)[0])
        lines[init] = series
        if init in m_idx:                      # the start's own lead-time climatology: the model's weak vortex
            c = m_idx[init]["clim"]["u60"].sel(time=slice(all_days[0], all_days[-1]))
            ax.plot(c["time"].values, c.values, color=color, lw=1.2, ls=":", alpha=0.8)
        for d in fc[init]["feat_ssw"]:         # SSWs of the featured member
            ax.plot([d], [0], marker="v", color=color, ms=9, mec="k", mew=0.6, zorder=6)
    e_part = e_u60.sel(time=slice(all_days[0], all_days[-1]))
    obs_line = ax.plot([], [], color="k", lw=FIG["obs_lw"], zorder=5)[0]
    cursor = ax.axvline(all_days[0], color="0.2", lw=1.2, ls="--", zorder=3)
    ax.set_xlim(all_days[0], all_days[-1])
    lo = min([-15.0] + [float(v[0].min()) for s in lines.values() for v in s.values()] + ([float(e_part.min())] if e_part.size else []))
    hi = max([45.0] + [float(v[0].max()) for s in lines.values() for v in s.values()] + ([float(e_part.max())] if e_part.size else []))
    ax.set_ylim(np.floor(lo / 5) * 5, np.ceil(hi / 5) * 5)
    ax.set_ylabel("U(60°N, 10 hPa) (m s$^{-1}$)", fontsize=FIG["label"])
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
    ax.tick_params(labelsize=FIG["tick"])
    ax.grid(alpha=0.25)
    handles = [Line2D([], [], color="k", lw=FIG["obs_lw"], label="ERA5"),
               Patch(color="0.85", label=f"ERA5 {CLIM_YEARS[0]}-{CLIM_YEARS[1]} ±1σ")]
    handles += [Line2D([], [], color=INIT_COLORS[i], lw=FIG["ens_lw"], label=f"1 {MONTH_NAMES[i]} start, mean") for i in inits]
    handles += [Line2D([], [], color="0.3", lw=FIG["member_lw"], ls="--", label="featured member"),
                Line2D([], [], color="0.3", lw=1.2, ls=":", label="E3SM lead-time climatology"),
                Line2D([], [], color="0.3", marker="v", ls="", mec="k", label="featured-member SSW")]
    ax.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=4, fontsize=FIG["legend"],
              frameon=False)
    counts = "; ".join(f"{MONTH_NAMES[i]} start {fc[i]['n_ssw']} of {fc[i]['n']}" for i in inits)
    ax.text(0.005, 0.97, f"members with an SSW in {WINTER_LABEL}: {counts}", transform=ax.transAxes, va="top",
            fontsize=FIG["legend"], bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.8})
    if any(fc[i]["surrogate"] for i in inits):
        fig.text(FIG["left"], 0.935, "† anomalies relative to the lead_drift climatology (no Reanalysis start in that month): "
                 "their sign is not robust to the drift correction (notebook 9, section 6)", fontsize=FIG["legend"] - 0.5, color="0.3")
    fig.suptitle(f"Stratospheric polar vortex, winter {WINTER_LABEL}: 10-hPa geopotential height", x=FIG["left"],
                 ha="left", y=0.975, fontsize=FIG["suptitle"], fontweight="bold")
    state.update(lines=lines, e_part=e_part, obs_line=obs_line, cursor=cursor,
                 day_text=fig.text(FIG["right"], 0.975, "", ha="right", va="top", fontsize=FIG["day"], fontweight="bold"))
    return fig, state


def draw_frame(state, t):
    t = pd.Timestamp(t)
    state["day_text"].set_text(f"{t:%d %B %Y}")
    for key, (ax, mstate) in state["maps"].items():
        if key == "obs":
            if obs_anom is None or value_at(obs_anom, t) is None:
                fill_map(ax, mstate, empty=f"Not yet observed\n(ERA5 ends {obs_end:%d %b %Y})")
            else:
                fill_map(ax, mstate, value_at(obs_anom, t), value_at(z_obs, t),
                         **u60_badge(float(e_u60.sel(time=t)) if t <= obs_end else None))
        elif key == "clim":
            fill_map(ax, mstate, None, value_at(z_clim, t), **u60_badge(float(u60_clim_days.sel(time=t)), "U60 clim"))
        else:
            kind, init = key
            f = fc[init]
            anom = value_at(f[f"{kind}_anom"], t)
            if anom is None:
                fill_map(ax, mstate, empty="Before start" if t < pd.Timestamp(YEAR, init, 1) else "No output")
                continue
            u = value_at(f["u60_mean" if kind == "mean" else "u60_feat"], t)
            fill_map(ax, mstate, anom, value_at(f[f"{kind}_total"], t), **u60_badge(None if u is None else float(u)))
    tn = np.datetime64(t, "ns")
    for init, series in state["lines"].items():
        for s, line in series.values():
            part = s.sel(time=slice(None, tn))
            line.set_data(part["time"].values, part.values)
    part = state["e_part"].sel(time=slice(None, tn))
    state["obs_line"].set_data(part["time"].values, part.values)
    state["cursor"].set_xdata([t, t])
    return []

### Preview: one frame

In [ ]:
preview_day = frame_days[len(frame_days) // 2]
for init in fc:                       # a featured member's SSW day, if any, is more telling
    if "first SSW" in fc[init]["why"]:
        preview_day = pd.Timestamp(fc[init]["why"].split("(")[1].split(")")[0])
        break
fig, state = build_figure()
draw_frame(state, preview_day)
plt.show()

### Animation

One GIF and one video under `figures/`, one frame every `FRAME_STEP` days from `FRAMES[0]` to
`FRAMES[1]`, the last frame held for `HOLD` frames.

In [ ]:
%%time
def video_writer():
    # (extension, writer factory) for the best video codec the available ffmpeg has, or (None, None)
    try:
        import imageio_ffmpeg   # bundled ffmpeg with H.264
        plt.rcParams["animation.ffmpeg_path"] = imageio_ffmpeg.get_ffmpeg_exe()
    except ImportError:
        pass
    if not animation.writers.is_available("ffmpeg"):
        return None, None
    encoders = subprocess.run([plt.rcParams["animation.ffmpeg_path"], "-hide_banner", "-encoders"],
                              capture_output=True, text=True).stdout
    even = ["-vf", "scale=trunc(iw/2)*2:trunc(ih/2)*2", "-pix_fmt", "yuv420p"]
    if "libx264" in encoders:
        return "mp4", lambda: animation.FFMpegWriter(fps=FPS, codec="libx264", extra_args=even + ["-crf", "20"])
    if "libvpx-vp9" in encoders:
        return "webm", lambda: animation.FFMpegWriter(fps=FPS, codec="libvpx-vp9", extra_args=even + ["-crf", "30", "-b:v", "0"])
    return None, None


video_ext, video = video_writer()
writers = {"gif": ("gif", lambda: animation.PillowWriter(fps=FPS)), "video": (video_ext, video)}
print(f"video: {video_ext or 'no ffmpeg video encoder; skipped'}")
sequence = list(frame_days) + [frame_days[-1]] * HOLD
fig, state = build_figure()
anim = animation.FuncAnimation(fig, lambda t: draw_frame(state, t), frames=sequence, blit=False, cache_frame_data=False)
anim_files = {}
for fmt in ANIM_FORMATS:
    ext, writer = writers[fmt]
    if writer is None:
        continue
    path = FIGDIR / figure_filename("RTENSO2026", f"winter{WINTER}{WINTER + 1}", "Z10", "polar_vortex", ext=ext)
    anim.save(path, writer=writer(), dpi=ANIM_DPI)
    anim_files[ext] = path
    print(f"{path} ({path.stat().st_size / 1e6:.1f} MB)")
plt.close(fig)
if SHOW_INLINE and "gif" in anim_files:
    display(Image(filename=str(anim_files["gif"])))

### Storyboard

The same panels for the `STORYBOARD` days in one static figure (rows: days; columns: ERA5, then each
start's ensemble mean and featured member), for papers and handouts.

In [ ]:
def storyboard():
    inits = [i for i in INIT_MONTHS if i in fc]
    columns = [("obs", "ERA5", "k")]
    for init in inits:
        columns += [(("mean", init), f"{MONTH_NAMES[init]} start mean", INIT_COLORS[init]),
                    (("feat", init), f"{MONTH_NAMES[init]} start, member {fc[init]['featured'].split(':')[-1]}",
                     INIT_COLORS[init])]
    rows = [pd.Timestamp(d) for d in STORYBOARD]
    fig, axes = plt.subplots(len(rows), len(columns), figsize=(2.3 * len(columns), 2.45 * len(rows) + 1.0),
                             subplot_kw={"projection": PROJ}, squeeze=False, gridspec_kw={"hspace": 0.22, "wspace": 0.04})
    for r, t in enumerate(rows):
        for c, (key, title, color) in enumerate(columns):
            ax = axes[r, c]
            mstate = setup_map(ax, title if r == 0 else "", color)
            mstate["badge"].set_fontsize(FIG["badge"] - 2)
            if key == "obs":
                if obs_anom is None or value_at(obs_anom, t) is None:
                    fill_map(ax, mstate, empty="Not yet\nobserved")
                else:
                    fill_map(ax, mstate, value_at(obs_anom, t), value_at(z_obs, t), **u60_badge(float(e_u60.sel(time=t))))
            else:
                kind, init = key
                anom = value_at(fc[init][f"{kind}_anom"], t)
                u = value_at(fc[init]["u60_mean" if kind == "mean" else "u60_feat"], t)
                fill_map(ax, mstate, anom, value_at(fc[init][f"{kind}_total"], t),
                         **u60_badge(None if u is None else float(u)), **({} if anom is not None else {"empty": "Before start"}))
            if c == 0:
                ax.text(-0.12, 0.5, f"{t:%d %b %Y}", transform=ax.transAxes, rotation=90, ha="center", va="center",
                        fontsize=FIG["label"], fontweight="bold")
    levels, cmap, norm = levels_cmap()
    cbar = fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=cmap), ax=axes, orientation="horizontal", extend="both",
                        fraction=0.02, pad=0.03, shrink=0.5, aspect=40,
                        ticks=np.arange(-STYLE["range"], STYLE["range"] + 1e-6, 4 * STYLE["ci"]))
    cbar.set_label(f"{STYLE['name']} ({STYLE['units']}); contours: total every {STYLE['line']:g} m", fontsize=FIG["label"])
    fig.suptitle(f"Polar vortex at 10 hPa, winter {WINTER_LABEL}: ERA5, E3SM ensemble means and featured members",
                 fontsize=FIG["suptitle"] - 2, y=0.92)
    return fig


fig = storyboard()
print(rc.save_figure(fig, NOTEBOOK, "RTENSO2026", f"winter{WINTER}{WINTER + 1}", "Z10", "storyboard", dpi=fig_dpi))
plt.show()